## Student Result Processor
--- 
### Problem Statement

The academic department maintains student result records in multiple CSV files stored inside a folder.
Some of these files may be missing, corrupted, or may contain invalid student data such as non-numeric marks or marks outside the valid range.

---
### Imports & Custom Exception


In [1]:
import os
import csv

class InvalidStudentDataError(Exception):
    pass

### Create Folder & Sample CSV Records

In [ ]:
folder = "student_records"
if not os.path.exists(folder):
    os.mkdir(folder)
    print(f"Folder '{folder}' created.")
else:
    print(f"Folder '{folder}' already exists.")
file_data = {
    "class_a.csv": [
        ["Name", "Roll No", "Marks"],
        ["Ali", "A01", 85],
        ["Sara", "A02", 35],      
        ["Usman", "A03", 105],    
        ["Zara", "A04", "abc"],   
    ],
    "class_b.csv": [
        ["Name", "Roll No", "Marks"],
        ["Ahmed", "B01", 55],
        ["Hina", "B02", 28],      
        ["Bilal", "B03", 72],
    ],
    "corrupted.csv": [
        ["Name", "Roll No", "Marks"],
        ["InvalidRow"],       
    ]
}
for filename, data in file_data.items():
    path = os.path.join(folder, filename)
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(data)
    print(f"Created: {filename}")

Folder 'student_records' created.
Created: class_a.csv
Created: class_b.csv
Created: corrupted.csv


### Student Result Processor

In [ ]:
all_results = []

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)

    try:
        if not filename.endswith(".csv"):
            continue

        with open(file_path, "r") as f:
            reader = csv.reader(f)
            headers = next(reader)  

            for row in reader:
                try:
                    name = row[0]
                    roll_no = row[1]
                    marks = float(row[2])

                    if marks < 0 or marks > 100:
                        raise InvalidStudentDataError(f"Invalid marks for {name}")

                    status = "Fail" if marks < 40 else "Pass"
                    all_results.append([name, roll_no, marks, status])

                except ValueError:
                    print(f"Invalid marks in {filename}: {row}")
                except IndexError:
                    print(f"Corrupted row in {filename}: {row}")
                except InvalidStudentDataError as e:
                    print(f"Data error: {e}")

    except FileNotFoundError:
        print(f"File not found: {filename}")
    except Exception as e:
        print(f"Error reading {filename}: {e}")
    finally:
        print(f"Finished processing: {filename}")

Data error: Invalid marks for Usman
Invalid marks in class_a.csv: ['Zara', 'A04', 'abc']
Finished processing: class_a.csv
Finished processing: class_b.csv
Corrupted row in corrupted.csv: ['InvalidRow']
Finished processing: corrupted.csv


### Write Clean Results to Output CSV

In [5]:
output_file = "clean_results.csv"

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Name", "Roll No", "Marks", "Status"])
    writer.writerows(all_results)

print(f"\nClean results saved in: {output_file}")


Clean results saved in: clean_results.csv


### View Clean Results

In [6]:
with open("clean_results.csv", "r") as f:
    print(f.read())

Name,Roll No,Marks,Status
Ali,A01,85.0,Pass
Sara,A02,35.0,Fail
Ahmed,B01,55.0,Pass
Hina,B02,28.0,Fail
Bilal,B03,72.0,Pass

